In [1]:
import pandas as pd
import numpy as np
import os
import warnings

In [2]:
warnings.filterwarnings('ignore')

In [3]:
# 1. 경로 및 데이터 로드
BASE_PATH = r'D:\AiSon\DTS\BDW'
print("🔥 [개별 상품 분석] 데이터 로드 중...")

orders = pd.read_csv(os.path.join(BASE_PATH, 'olist_orders_dataset.csv'))
items = pd.read_csv(os.path.join(BASE_PATH, 'olist_order_items_dataset.csv'))
customers = pd.read_csv(os.path.join(BASE_PATH, 'olist_customers_dataset.csv'))
sellers = pd.read_csv(os.path.join(BASE_PATH, 'olist_sellers_dataset.csv'))
reviews = pd.read_csv(os.path.join(BASE_PATH, 'olist_order_reviews_dataset.csv'))
geo = pd.read_csv(os.path.join(BASE_PATH, 'olist_geolocation_dataset.csv'))
products = pd.read_csv(os.path.join(BASE_PATH, 'olist_products_dataset.csv'))
trans = pd.read_csv(os.path.join(BASE_PATH, 'product_category_name_translation.csv'))

🔥 [개별 상품 분석] 데이터 로드 중...


In [4]:
# 2. 병합 및 거리 계산
df = orders.merge(items, on='order_id')
df = df.merge(customers[['customer_id', 'customer_zip_code_prefix']], on='customer_id')
df = df.merge(sellers[['seller_id', 'seller_zip_code_prefix']], on='seller_id')
df = df.merge(reviews[['order_id', 'review_score']], on='order_id')
products = products.merge(trans, on='product_category_name', how='left')
df = df.merge(products[['product_id', 'product_category_name_english']], on='product_id', how='left')

In [5]:
geo_avg = geo.groupby('geolocation_zip_code_prefix')[['geolocation_lat', 'geolocation_lng']].mean().reset_index()
df = df.merge(geo_avg, left_on='customer_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='inner').rename(columns={'geolocation_lat': 'c_lat', 'geolocation_lng': 'c_lng'})
df = df.merge(geo_avg, left_on='seller_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='inner').rename(columns={'geolocation_lat': 's_lat', 'geolocation_lng': 's_lng'})

In [6]:
R = 6371
phi1, phi2 = np.radians(df['c_lat']), np.radians(df['s_lat'])
dphi = np.radians(df['s_lat'] - df['c_lat'])
dlambda = np.radians(df['s_lng'] - df['c_lng'])
a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2) * np.sin(dlambda/2)**2
df['distance_km'] = R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

In [7]:
# -------------------------------------------------------------------------
# 3. [핵심 변경] 카테고리가 아니라 'Product_ID' 기준으로 집계
# -------------------------------------------------------------------------
print("🔥 [분석 중] 수만 개의 개별 상품(Product ID)을 전수 조사합니다...")

item_stats = df.groupby(['product_id', 'product_category_name_english']).agg({
    'order_id': 'count',       # 판매량
    'review_score': 'mean',    # 평균 평점
    'distance_km': 'mean',     # 평균 배송 거리
    'price': 'mean'            # 평균 가격
}).reset_index()

item_stats.columns = ['Product_ID', 'Category', 'Volume', 'Score', 'Distance', 'Price']

🔥 [분석 중] 수만 개의 개별 상품(Product ID)을 전수 조사합니다...


In [8]:
# 신뢰도 필터: 최소 5개 이상은 팔린 물건만 분석 (우연히 1개 팔리고 5점 받은거 제외)
valid_items = item_stats[item_stats['Volume'] >= 5]

# 전체 평균 거리 계산
avg_dist = valid_items['Distance'].mean()

In [9]:
# -------------------------------------------------------------------------
# 4. 전략별 개별 상품 추출
# -------------------------------------------------------------------------

# [Hero 상품] 거리 > 평균 & 평점 >= 4.5 (진짜 명품)
hero_items = valid_items[
    (valid_items['Distance'] > avg_dist) & 
    (valid_items['Score'] >= 4.5)
].sort_values(by='Volume', ascending=False).head(5)

In [10]:
# [Cash Cow 상품] 거리 < 평균 & 평점 >= 4.3 (많이 팔리는 효자)
cash_cow_items = valid_items[
    (valid_items['Distance'] < avg_dist) & 
    (valid_items['Score'] >= 4.3)
].sort_values(by='Volume', ascending=False).head(5)

In [15]:
# -------------------------------------------------------------------------
# 5. 결과 출력 (ID 앞 8자리만 보여줌 - 가독성 위해)
# -------------------------------------------------------------------------
print("\n" + "="*80)
print(f"🚀 [전략 1] Hero 개별 상품 TOP 5 (장거리 고만족)")
print("   * Product_ID는 실제 상품 고유 코드입니다.")
print("="*80)
# 가독성을 위해 ID를 앞 10자리만 자르고 출력
hero_display = hero_items.copy()
hero_display['Product_ID'] = hero_display['Product_ID'].str[:10] + "..."
print(hero_display[['Category', 'Product_ID', 'Score', 'Distance', 'Volume', 'Price']].to_string(index=False))


🚀 [전략 1] Hero 개별 상품 TOP 5 (장거리 고만족)
   * Product_ID는 실제 상품 고유 코드입니다.
             Category    Product_ID    Score    Distance  Volume      Price
           cool_stuff 5f504b3a1c... 4.555556  773.807212      63 598.950794
          electronics 6a8631b72a... 4.709677  787.329773      62  26.641935
           housewares d696750e55... 4.526316  682.305262      57 172.944912
computers_accessories 130482add9... 4.538462 1008.787072      52 105.000000
                 food ed2067a9c1... 4.538462  599.969201      52  58.011346


In [16]:
print("\n" + "="*80)
print(f"💰 [전략 2] Cash Cow 개별 상품 TOP 5 (근거리 고매출)")
print("   * 이 상품들은 재고 관리 1순위 대상입니다.")
print("="*80)
cow_display = cash_cow_items.copy()
cow_display['Product_ID'] = cow_display['Product_ID'].str[:10] + "..."
print(cow_display[['Category', 'Product_ID', 'Score', 'Distance', 'Volume', 'Price']].to_string(index=False))


💰 [전략 2] Cash Cow 개별 상품 TOP 5 (근거리 고매출)
   * 이 상품들은 재고 관리 1순위 대상입니다.
      Category    Product_ID    Score   Distance  Volume      Price
 health_beauty 154e7e31eb... 4.307958 584.035957     289  22.509031
bed_bath_table f1c7f35307... 4.384106 569.118745     151 194.552053
 health_beauty e0cf79767c... 4.481481 272.313518     135  29.900000
    cool_stuff 54d9ac713e... 4.350877 310.127552     114  32.074561
     perfumery 595fac2a38... 4.405660 522.177600     106 119.216038


1. Product_ID

"Olist 데이터베이스에 등록된 실제 상품의 주민등록번호(SKU)입니다.

표의 맨 위를 봐주십시오. 5f504b... 라는 코드를 가진 상품이 있습니다. 이 상품은 '쿨 스터프(Cool Stuff)' 카테고리인데, 가격이 약 599헤알(약 14만원)로 꽤 비쌉니다.
놀라운 점은, 고객들이 이 비싼 물건을 받으려고 평균 773km(서울-부산 왕복 거리)를 기다렸고, 평점은 4.55점이나 줬다는 사실입니다.

즉, 이 ID(5f504b...)를 가진 상품은 거리가 멀어도, 비싸도 팔리는 우리 회사의 '슈퍼스타'입니다."

2. 카테고리와 차이점

"아까 보여드린 게 'Category'라는 숲(전체)을 본 거라면, 지금 보시는 건 그 숲에서 가장 높게 자란 나무(특정 인기 상품)를 찍어낸 것입니다.

마케팅 팀에 '쿨 스터프 광고해'라고 하면 막막해하지만, '이 상품코드(5f504b...)를 메인 배너에 걸어'라고 하면 즉시 실행할 수 있습니다. 우리가 찾은 건 바로 그 구체적인 실행 타겟입니다."